# Trino + S3 parquet


In [1]:
import time
import pandas as pd
from sqlalchemy import create_engine, text

trino = create_engine("trino://trino@trino:8080/hive")

def wait_for_engine(engine, attempts=24, pause=5):
    last_error = None
    for _ in range(attempts):
        try:
            pd.read_sql("SELECT 1", engine)
            return
        except Exception as error:
            last_error = error
            time.sleep(pause)
    raise RuntimeError("Trino is not ready") from last_error

wait_for_engine(trino)


In [2]:
statements = [
    "CREATE SCHEMA IF NOT EXISTS hive.oil",
    "DROP TABLE IF EXISTS hive.oil.production_parquet",
    '''
    CREATE TABLE hive.oil.production_parquet (
        prod_id integer,
        well_id integer,
        date date,
        oil_ton double,
        gas_m3 double,
        water_m3 double,
        energy_kwh double,
        downtime_hours double,
        temperature double,
        pressure double,
        dt date
    )
    WITH (
        external_location = 's3://oil-lake/bronze/production',
        format = 'PARQUET',
        partitioned_by = ARRAY['dt']
    )
    ''',
    "CALL hive.system.sync_partition_metadata('oil', 'production_parquet', 'FULL')",
]
with trino.begin() as connection:
    for statement in statements:
        connection.execute(text(statement))


In [3]:
pd.read_sql("SELECT table_schema, table_name FROM hive.information_schema.tables WHERE table_schema = 'oil' ORDER BY table_name", trino)


,table_schema,table_name
0,oil,production_parquet


In [4]:
pd.read_sql("SELECT dt, count(*) AS rows_count, round(sum(oil_ton), 2) AS oil_ton FROM hive.oil.production_parquet GROUP BY dt ORDER BY dt LIMIT 10", trino)


,dt,rows_count,oil_ton
0,2025-10-01,10,1435.0
1,2025-10-02,10,1434.6
2,2025-10-03,10,1438.4
3,2025-10-04,10,1445.8
4,2025-10-05,10,1442.0
5,2025-10-06,10,1437.2
6,2025-10-07,10,1428.2
7,2025-10-08,10,1432.8
8,2025-10-09,10,1443.4
9,2025-10-10,10,1437.0
